In [42]:
import json
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

verbose = 0
data_set = "gmtkn"

subset = "molecule_W4_11 molecule_G21EA molecule_G21IP molecule_DIPCS10 molecule_PA26 molecule_SIE4x4 molecule_ALKBDE10 molecule_YBDE18 molecule_AL2X6 molecule_HEAVYSB11 molecule_NBPRC molecule_ALK8 molecule_RC21 molecule_G2RC molecule_BH76 molecule_FH51 molecule_TAUT15 molecule_DC13".split(
    " "
)
subset_list = [x.split("molecule_")[1] for x in subset]
print(subset_list)

data_path_list = sorted(
    list(Path("../validate").glob(f"*{data_set}.csv")),
    key=lambda p: p.stat().st_ctime,
)
basis_args = "cc-pVDZ"
print(basis_args)

with open(f"../cc2cc/utils/{data_set}.json") as f:
    json_data = json.load(f)

# accumulate summary dictionaries for each file

for data_path in data_path_list:
    summary_list = []

    data = pd.read_csv(data_path)
    data["name"] = data["name"].str.split(f"_{basis_args}").str[0]

    for i_subset in subset_list:
        data_name = []
        data_reaction_energy_dft = []
        data_reaction_energy_ai = []

        reaction_dict = json_data[f"reaction-{i_subset}"]
        reaction_dict_copy = reaction_dict.copy()
        for i_reaction_name, i_reaction in reaction_dict_copy.items():
            systems_list = i_reaction["systems"]
            stoichiometry_list = i_reaction["stoichiometry"]

            atomic_energy_dft = 0
            atomic_energy_ai = 0
            for i in range(len(systems_list)):
                finished = True
                mole_name = f"{i_subset}-{systems_list[i]}"

                if mole_name in json_data:
                    if isinstance(json_data[mole_name], str):
                        mole_name = json_data[mole_name]
                else:
                    finished = False
                    reaction_dict.pop(i_reaction_name)
                    break

                col = data["name"] == mole_name
                if col.any():
                    atomic_energy_dft += data[col]["error_dft_ene"].values[0] * int(
                        stoichiometry_list[i]
                    )
                    atomic_energy_ai += data[col]["error_scf_ene"].values[0] * int(
                        stoichiometry_list[i]
                    )
                    if verbose == 2:
                        print(
                            data[col]["error_dft_ene"].values[0],
                            int(stoichiometry_list[i]),
                            systems_list[i],
                        )
                else:
                    finished = False
                    break

            if finished:
                data_reaction_energy_dft.append(atomic_energy_dft)
                data_reaction_energy_ai.append(atomic_energy_ai)
                data_name.append(i_reaction_name)

        data_name = np.array(data_name)
        data_reaction_energy_dft = np.array(data_reaction_energy_dft)
        data_reaction_energy_ai = np.array(data_reaction_energy_ai)

        if verbose == 1:
            dft_error_argsort = np.argsort(data_reaction_energy_dft)[:5]
            ai_error_argsort = np.argsort(data_reaction_energy_ai)[:5]

            print("====DFT error====")
            print(
                [
                    json_data[f"reaction-{i_subset}"][data_name[i]]["systems"]
                    for i in dft_error_argsort
                ]
            )
            print(data_reaction_energy_dft[dft_error_argsort])
            print("====AI error====")
            print(
                [
                    json_data[f"reaction-{i_subset}"][data_name[i]]["systems"]
                    for i in ai_error_argsort
                ]
            )
            print(data_reaction_energy_ai[ai_error_argsort])

        summary = {
            "subset": i_subset,
            "AI AE": f"{np.mean(np.abs(data_reaction_energy_ai) if len(data_reaction_energy_ai) else 0):.2f}",
            "DFT AE": f"{np.mean(np.abs(data_reaction_energy_dft) if len(data_reaction_energy_dft) else 0):.2f}",
            "Processed": f"{len(data_reaction_energy_dft)} / {len(reaction_dict)}",
        }
        summary_list.append(summary)

    # display one summary table for all files
    df_summary = pd.DataFrame(summary_list)
    print(f"Summary of Atomic Energies of {data_path.stem}:")
    display(df_summary)

['W4_11', 'G21EA', 'G21IP', 'DIPCS10', 'PA26', 'SIE4x4', 'ALKBDE10', 'YBDE18', 'AL2X6', 'HEAVYSB11', 'NBPRC', 'ALK8', 'RC21', 'G2RC', 'BH76', 'FH51', 'TAUT15', 'DC13']
cc-pVDZ
Summary of Atomic Energies of ccdft_cc-pVDZ_atom-1-2272051_gmtkn:


,subset,AI AE,DFT AE,Processed
0,W4_11,2.97,29.51,140 / 140
1,G21EA,2.14,9.75,25 / 25
2,G21IP,0.68,1.24,1 / 36
3,DIPCS10,0.00,0.00,0 / 10
4,PA26,0.00,0.00,0 / 26
5,SIE4x4,0.00,0.00,0 / 16
6,ALKBDE10,0.00,0.00,0 / 9
7,YBDE18,0.00,0.00,0 / 18
8,AL2X6,0.00,0.00,0 / 6
9,HEAVYSB11,0.00,0.00,0 / 7


Summary of Atomic Energies of ccdft_cc-pVDZ_atom-1-3052180_gmtkn:


,subset,AI AE,DFT AE,Processed
0,W4_11,1.74,13.53,14 / 140
1,G21EA,0.58,10.24,7 / 25
2,G21IP,0.90,10.14,15 / 36
3,DIPCS10,1.21,14.32,2 / 10
4,PA26,0.00,0.00,0 / 26
5,SIE4x4,17.89,22.18,4 / 16
6,ALKBDE10,7.95,21.11,5 / 9
7,YBDE18,0.00,0.00,0 / 18
8,AL2X6,0.00,0.00,0 / 6
9,HEAVYSB11,2.42,10.09,2 / 7


In [23]:
json_data[f"reaction-{i_subset}"][data_name[0]]["systems"]

['h2', 'h']

In [2]:
import numpy as np
(
    np.array([10.58423375510182, 225.43998315640636])
    - np.array([2.4392066220511355, 212.98927779812027])
)

# NBPRC-nh3-bh3

array([ 8.14502713, 12.45070536])

|File | AI AE | DFT AE | AI E | DFT E | AI Ele | DFT Ele | AI Dip | DFT Dip | Processed |
|---|---|---|---|---|---|---|---|---|---|
| atom-1-4049491 | 3.18 | 29.85 | 2.99 | 320.81 | 0.11 | 0.16 | 0.027 | 0.023 | 140 / 140 |
| atom-1-4049491 | 1.98 | 28.07 | 1.90 | 291.66 | 0.11 | 0.15 | 0.029 | 0.025 | 128 / 140 |
| atom-1-4049491 | 1.33 | 29.85 | 1.52 | 320.81 | 0.11 | 0.16 | 0.028 | 0.023 | 140 / 140 |
